# Finding an edge by hand

This notebook goes with the Detailed lesson **Computer Vision: Pixels, Filters and Convolutional Networks**, section 5.

We store a tiny image as numbers, slide filters over it, and pool the result. Run each cell in order with **Shift + Enter**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

image = np.array([[0, 0, 0, 9, 9]] * 4)   # 4 rows, 5 columns: dark on the left, bright on the right
print(image)

## 1. Two edge filters

In [ ]:
vertical = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]])
horizontal = vertical.T   # the same weights turned on their side
print("vertical-edge filter:\n", vertical)
print("horizontal-edge filter:\n", horizontal)

## 2. Convolution

At each position, multiply the nine pixels under the filter by its weights and add them up. With no padding, the output is (rows − 2) × (columns − 2).

In [ ]:
def convolve(img, kernel, show=False):
    k = kernel.shape[0]
    rows, cols = img.shape[0] - k + 1, img.shape[1] - k + 1
    out = np.zeros((rows, cols), dtype=int)
    for r in range(rows):
        for c in range(cols):
            window = img[r:r + k, c:c + k]
            out[r, c] = int(np.sum(window * kernel))
            if show and r == 0:
                print(f"position (row {r}, col {c}): window rows {window.tolist()} -> {out[r, c]}")
    return out

v = convolve(image, vertical, show=True)
print()
print("vertical-edge feature map:\n", v)
print("horizontal-edge feature map:\n", convolve(image, horizontal))

## 3. Max pooling

Keep the largest value in each 2 × 2 block. The edge is still recorded, but not its exact position.

In [ ]:
def max_pool(fm, size=2):
    rows, cols = fm.shape[0] // size, fm.shape[1] // size
    return np.array([[fm[r*size:(r+1)*size, c*size:(c+1)*size].max() for c in range(cols)] for r in range(rows)])

print(max_pool(v))

## 4. A bigger drawing

A 6 × 6 image gives a 4 × 4 feature map, as in the lesson's Filter Lab.

In [ ]:
cross = np.zeros((6, 6), dtype=int)
cross[2:4, :] = 9
cross[:, 2:4] = 9
blur = np.ones((3, 3), dtype=int)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (title, data) in zip(axes, [("image", cross),
                                    ("vertical edges", convolve(cross, vertical)),
                                    ("horizontal edges", convolve(cross, horizontal)),
                                    ("blur (sum / 9)", convolve(cross, blur) / 9)]):
    ax.imshow(data, cmap="gray")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 5. Try it yourself

- Design a 3 × 3 filter that responds to a diagonal edge, and test it on a diagonal line.
- Use a stride of 2 (move the window two pixels at a time). What size is the output?